# Creación BD S&P500

La idea es que del dataset con los precios, se tome el último día y se resten 1000 días, después se compare con el otro dataset de tickers para evaluar el sesgo de supervivencia. Los enlaces a dichos datasets se encuentran a continuación:

Dataset con la información de precios: https://www.kaggle.com/datasets/camnugent/sandp500?resource=download

Dataset con la información de tickers: https://github.com/fja05680/sp500

In [60]:
import pandas as pd
import os
import numpy as np
import pandas_market_calendars as mcal
import matplotlib.pyplot as plt


In [61]:
# ---------------------------------------------------------
# BLOQUE 1: Configuración de Rutas y Carga Inicial
# ---------------------------------------------------------
folder_path = r'../../../data/processed'
path_main = os.path.join(folder_path, 'dataset_with_returns.csv')
path_components = os.path.join(folder_path, 'S&P 500 Historical Components & Changes(01-17-2026).csv')

print("Cargando dataset principal...")
df = pd.read_csv(path_main)
df['date'] = pd.to_datetime(df['date'])

# Aseguramos el orden cronológico para que el recorte de registros sea correcto
df = df.sort_values(['Name', 'date'])

df.head()

Cargando dataset principal...


,date,open,high,low,close,volume,Name,adjusted_price,return_1d
0,2013-02-11,45.17,45.18,44.45,44.60,2915405,A,28.576422,-0.010648
1,2013-02-12,44.81,44.95,44.50,44.62,2373731,A,28.589230,0.000448
2,2013-02-13,44.81,45.24,44.68,44.75,2052338,A,28.672525,0.002914
3,2013-02-14,44.72,44.78,44.36,44.58,3826245,A,28.563612,-0.003799
4,2013-02-15,43.48,44.24,42.21,42.25,14657315,A,27.070713,-0.052266


In [62]:
df['date'] = pd.to_datetime(df['date'])

print("Fecha inicial:", df['date'].min())
print("Fecha final:", df['date'].max())

Fecha inicial: 2013-02-11 00:00:00
Fecha final: 2018-02-07 00:00:00


### Checking if there are dates gaps

In [ ]:
def detect_real_gaps(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["Name", "date"])

    nyse = mcal.get_calendar("NYSE")

    gaps = []

    for name, group in df.groupby("Name"):
        group = group.sort_values("date").reset_index(drop=True)

        for i in range(1, len(group)):
            prev_date = group.loc[i-1, "date"]
            curr_date = group.loc[i, "date"]

            valid_days = nyse.valid_days(start_date=prev_date, end_date=curr_date)

            # deberían ser 2: prev y next
            if len(valid_days) > 2:
                gaps.append({
                    "Name": name,
                    "prev_date": prev_date,
                    "date": curr_date,
                    "missing_days": valid_days[1:-1]
                })

    return pd.DataFrame(gaps)
    
detect_real_gaps(df).shape

### Avoiding survivor bias

To eliminate survivor bias, we first obtain all month-end constituent lists 
for the S&P 500 from January 2013 to December 2018. We consolidate these lists 
into a single binary matrix, indicating whether each stock is an index 
constituent in the subsequent month. In this way, we are able to approximately 
reproduce the S&P 500 composition at any point in time within this period. 
In a second step, for all stocks that have been constituents of the index at 
any time during this interval, we download daily total return indices covering 
the same time frame.

In [ ]:
folder_path = r'../../../data/raw'
path_components = os.path.join(folder_path, 'S&P 500 Historical Components & Changes(01-17-2026).csv')
df_hist_comp = pd.read_csv(path_components)
df_hist_comp['date'] = pd.to_datetime(df_hist_comp['date'])

print(df_hist_comp.head())
print("...")
print(df_hist_comp.tail())

In [ ]:
df_hist_comp.dtypes

In [ ]:
df.head()
first_date = df.iloc[0]['date']
print(first_date)

last_date = df.iloc[-1]['date']
print(last_date)

Creating pandas tables to see which companies were on S&P 500 at the end of the month

In [ ]:
# Filtrar por fechas
df_hist_comp = df_hist_comp[(df_hist_comp['date'] >= first_date) & (df_hist_comp['date'] <= last_date)].copy()

# Extraer año y mes
df_hist_comp['year'] = df_hist_comp['date'].dt.year
df_hist_comp['month'] = df_hist_comp['date'].dt.month

# Tomar último día de cada mes
df_month_end = df_hist_comp.groupby(['year', 'month']).tail(1)

# Eliminar la columna 'date'
df_month_end = df_month_end.drop(columns=['date'])

# Reordenar columnas para que year y month queden primero
cols = ['year', 'month'] + [c for c in df_month_end.columns if c not in ['year', 'month']]
df_month_end = df_month_end[cols]

df_month_end = df_month_end.reset_index(drop=True)

# Revisar resultado
print(df_month_end.head())
print("...")
print(df_month_end.tail())

Creating Binary matrix

In [ ]:
all_tickers = set()

for tickers_str in df_hist_comp['tickers']:
    for t in tickers_str.split(','):
        all_tickers.add(t.strip())
all_tickers = sorted(all_tickers)

ticker_dict = {ticker: idx for idx, ticker in enumerate(all_tickers)}


n_months = len(df_month_end)
n_tickers = len(all_tickers)

# matriz de ceros
binary_matrix = np.zeros((n_months, n_tickers), dtype=int)

for i, row in df_month_end.iterrows():
    tickers_in_row = row['tickers'].split(',')  # separa los tickers
    for t in tickers_in_row:
        t = t.strip()  # quitar espacios
        if t in ticker_dict:  # seguridad
            j = ticker_dict[t]  # columna correspondiente
            binary_matrix[i, j] = 1


binary_df = pd.DataFrame(binary_matrix, columns=all_tickers)
binary_df.insert(0, 'year', df_month_end['year'])
binary_df.insert(1, 'month', df_month_end['month'])

print(binary_df.head())

# Normalize data

In [ ]:
data = df.copy()

# fecha a datetime
data['date'] = pd.to_datetime(data['date'])

# ordenar
data = data.sort_values(['Name', 'date'])

# columnas OHLCV
features = ['open', 'high', 'low', 'close', 'volume']

# percentage change normalization
for col in features:
    data[col] = (
        data.groupby('Name')[col]
        .pct_change() * 100
    )

# eliminar NaNs creados por pct_change
data = data.dropna().reset_index(drop=True)

print(data.head())

In [ ]:
# eliminar columnas innecesarias
data = data.drop(columns=['adjusted_price', 'return_1d'])

# verificar
print(data.head())

In [ ]:
# tamaño figura
plt.figure(figsize=(10,6))

# histograma de returns del close
plt.hist(data['close'], bins=100)

# títulos
plt.title('Distribution of Close Price Percentage Changes')
plt.xlabel('Daily Close Return (%)')
plt.ylabel('Frequency')

# grid
plt.grid(True)

plt.show()

# Create sequences

## Cleaning data

### Filtrar tickers usando la matriz binaria

This function is made so it search the companies that were on S&P500 the last month of the 1000 blocks day

In [ ]:
def get_tickers_for_period(binary_df, last_train_date):
    year = last_train_date.year
    month = last_train_date.month
    
    row = binary_df[
        (binary_df['year'] == year) &
        (binary_df['month'] == month)
    ]
    
    if row.empty:
        return []
    
    tickers_series = row.drop(columns=['year', 'month']).iloc[0]
    tickers = tickers_series[tickers_series == 1].index.tolist()
    
    return tickers

final_date = df['date'].max()
tickers_to_use = get_tickers_for_period(binary_df, final_date)
print(len(tickers_to_use))

data = df[
    df['Name'].isin(tickers_to_use)
].copy()

print(data.shape)
print(data['Name'].nunique())


In [ ]:
ticker_counts = data.groupby('Name').size()

print("\nDistribución de cantidad de filas por ticker:")
print(ticker_counts.describe())

# mínimo recomendado
MIN_DAYS = 252

valid_tickers = ticker_counts[
    ticker_counts >= MIN_DAYS
].index

data = data[
    data['Name'].isin(valid_tickers)
].reset_index(drop=True)

print("\nTickers después de filtrar por mínimo de días:")
print(len(valid_tickers))

### Revisar gaps temporales

In [ ]:
print("\nRevisando gaps temporales...")

tickers_without_large_gaps = []

MAX_ALLOWED_GAP = 10

for ticker in data['Name'].unique():

    temp = data[data['Name'] == ticker].sort_values('date')

    gaps = temp['date'].diff().dt.days

    max_gap = gaps.max()

    if pd.isna(max_gap):
        continue

    if max_gap <= MAX_ALLOWED_GAP:
        tickers_without_large_gaps.append(ticker)

print("Tickers sin gaps grandes:",
      len(tickers_without_large_gaps))

data = data[
    data['Name'].isin(tickers_without_large_gaps)
].reset_index(drop=True)



### Delete extrem outliers

In [ ]:
print("\nEstadísticas antes de tratar outliers:")
print(data[features].describe())

# features de precios
price_features = ['open', 'high', 'low', 'close']

# threshold razonable para returns %
PRICE_CLIP_THRESHOLD = 50

# clip de returns extremos en precios
for col in price_features:

    data[col] = data[col].clip(
        lower=-PRICE_CLIP_THRESHOLD,
        upper=PRICE_CLIP_THRESHOLD
    )

# volume suele explotar mucho más
# usar threshold más amplio
VOLUME_CLIP_THRESHOLD = 500

data['volume'] = data['volume'].clip(
    lower=-VOLUME_CLIP_THRESHOLD,
    upper=VOLUME_CLIP_THRESHOLD
)

print("\nEstadísticas después del clipping:")
print(data[features].describe())

print("\nShape después del clipping:")
print(data.shape)

In [ ]:
print(data.isna().sum())

In [ ]:
data = data.dropna().reset_index(drop=True)

In [ ]:
print(data.isna().sum())

## Create sequences

In [ ]:
import numpy as np

features = ['open', 'high', 'low', 'close', 'volume']

# longitud de secuencia
SEQ_LEN = 20

X = []
y = []

# fechas target de cada secuencia
sequence_dates = []

# opcional: guardar ticker de cada sample
sequence_tickers = []

# Creación de secuencias
tickers_to_use = data['Name'].unique()

for ticker in tickers_to_use:

    # datos del ticker
    ticker_df = data[
        data['Name'] == ticker
    ].sort_values('date')

    # valores numpy
    values = ticker_df[features].values

    # fechas
    dates = ticker_df['date'].values

    # evitar tickers muy pequeños
    if len(values) <= SEQ_LEN:
        continue

    for i in range(len(values) - SEQ_LEN):

        # input sequence
        seq_x = values[i:i+SEQ_LEN]

        # target = close del siguiente día
        target_y = values[i+SEQ_LEN][3]

        # fecha correspondiente al target
        target_date = dates[i+SEQ_LEN]

        # guardar
        X.append(seq_x)
        y.append(target_y)

        sequence_dates.append(target_date)

        sequence_tickers.append(ticker)


X = np.array(X, dtype=np.float32)

y_regression = np.array(y, dtype=np.float32)

sequence_dates = np.array(sequence_dates)

sequence_tickers = np.array(sequence_tickers)

# INFO
print("X shape:", X.shape)
print("y shape:", y_regression.shape)

print("sequence_dates shape:", sequence_dates.shape)
print("sequence_tickers shape:", sequence_tickers.shape)

print("\nPrimera fecha:")
print(sequence_dates[0])

print("\nÚltima fecha:")
print(sequence_dates[-1])

## Classification targets

In [ ]:
targets_df = pd.DataFrame({

    'date': sequence_dates,
    'ticker': sequence_tickers,
    'return': y_regression

})

In [ ]:
targets_df['cross_sectional_median'] = (

    targets_df
    .groupby('date')['return']
    .transform('median')

)

In [ ]:
targets_df['class_target'] = (

    targets_df['return']
    >= targets_df['cross_sectional_median']

).astype(np.int8)

In [ ]:
y_classification = targets_df['class_target'].values.astype(np.int8)

In [ ]:
y_classification

In [ ]:
counts = pd.Series(y_classification).value_counts().sort_index()

plt.figure(figsize=(6, 4))

plt.bar(
    counts.index.astype(str),
    counts.values
)

plt.xlabel("Clase (target)")
plt.ylabel("Cantidad")
plt.title("Distribución de clases")

plt.tight_layout()
plt.show()

# Save data

In [ ]:

np.savez_compressed(

    'financial_dataset.npz',

    # inputs
    X=X,

    # targets
    y_regression=y_regression,
    y_classification=y_classification,

    # metadata
    sequence_dates=sequence_dates,
    sequence_tickers=sequence_tickers
)

print("Dataset guardado correctamente")